In [2]:
import numpy as np

# RNN FROM SCRATCH - TRADUCTION FR -> EN

# Ce programme montre le fonctionnement d'un réseau de neurones récurrent (RNN) pour traduire de petites
# phrases du français vers l'anglais.

# Jeu de données
# Chaque élément contient :
# (phrase française, traduction anglaise)

dataset = [

("bonjour","hello"),
("merci","thanks"),
("oui","yes"),
("non","no"),

("bonne nuit","good night"),

("je mange","i eat"),
("je cours","i run"),
("je dors","i sleep"),

("chat noir","black cat"),
("chien blanc","white dog")

]


# ==========================================================
# Construction du vocabulaire
#
# Le RNN ne comprend pas directement les mots.
# Chaque mot doit être transformé en un indice, puis en vecteur One-Hot.
fr_words = set()
en_words = set()

for fr, en in dataset:

    fr_words.update(fr.split())
    en_words.update(en.split())

# Tri des mots pour obtenir toujours le même ordre
fr_words = sorted(list(fr_words))
en_words = sorted(list(en_words))

# Création des dictionnaires

fr_word_to_index = {
    word: i
    for i, word in enumerate(fr_words)
}

en_word_to_index = {
    word: i
    for i, word in enumerate(en_words)
}


# Dictionnaire inverse
index_to_en_word = {
    i: word
    for word, i in en_word_to_index.items()
}


# Taille des vocabulaires
input_size = len(fr_words)
output_size = len(en_words)


print("Nombre de mots français :", input_size)
print("Nombre de mots anglais  :", output_size)

# Fonction : one_hot()
#
# Cette fonction transforme un mot en vecteur One-Hot.
# Exemple :
# Vocabulaire :
# bonjour -> 0
# merci   -> 1
# oui     -> 2
#
# "merci" devient : [0 1 0]

def one_hot(index, vocab_size):

    vector = np.zeros((vocab_size, 1))

    vector[index] = 1

    return vector


# Initialisation des paramètres du RNN
# Les poids sont initialisés avec de petites valeurs
# aléatoires.
#
# Wxh : entrée  → état caché
# Whh : état caché précédent → nouvel état caché
# Whz : état caché → sortie

hidden_size = 16      # Nombre de neurones dans l'état caché

Wxh = np.random.randn(hidden_size, input_size) * 0.01

Whh = np.random.randn(hidden_size, hidden_size) * 0.01

Whz = np.random.randn(output_size, hidden_size) * 0.01

# Initialisation des biais
# Les biais sont initialisés à zéro.

bh = np.zeros((hidden_size, 1))
bz = np.zeros((output_size, 1))

# Fonction : tanh()
# Cette fonction d'activation est utilisée pour calculer
# le nouvel état caché du RNN.
# La sortie est toujours comprise entre -1 et 1.

def tanh(x):

    return np.tanh(x)


# Fonction : softmax()
# La couche de sortie produit des scores.
# Softmax transforme ces scores en probabilités.
# La probabilité la plus élevée correspond au mot prédit.
def softmax(x):

    # Améliore la stabilité numérique
    x = x - np.max(x)

    # Calcul des exponentielles
    exp = np.exp(x)

    # Normalisation
    return exp / np.sum(exp)

# Fonction : rnn_cell()
#
# Cette fonction représente une cellule du RNN.
# Elle reçoit :
# - le mot courant x_t
# - l'état caché précédent h_prev
# Puis elle calcule :
# - le nouvel état caché h_t
# - la sortie z_t

def rnn_cell(x_t, h_prev):

    # Calcul du nouvel état caché
    h = tanh(np.dot(Wxh, x_t) + np.dot(Whh, h_prev) + bh)

    # Calcul des scores de sortie
    z = np.dot(Whz, h) + bz

    # Transformation des scores en probabilités
    y = softmax(z)

    return h, y


# Fonction : forward()
# Cette fonction réalise la propagation avant (Forward).
#
# Une phrase est traitée mot par mot.
# Pour chaque mot :
# - transformation en One-Hot
# - calcul du nouvel état caché
# - calcul de la sortie
#
# Les états cachés et les sorties sont sauvegardés
# afin de pouvoir effectuer la rétropropagation.

def forward(sentence):

    # Découpage de la phrase en mots
    words = sentence.split()

    # Etat caché initial
    h = np.zeros((hidden_size, 1))

    # Liste des états cachés
    hidden_states = []

    # Liste des sorties
    outputs = []

    # Parcours de tous les mots
    for word in words:

        # Numéro du mot dans le vocabulaire
        index = fr_word_to_index[word]

        # Transformation en vecteur One-Hot
        x = one_hot(index, input_size)

        # Passage dans la cellule RNN
        h, y = rnn_cell(x, h)

        # Sauvegarde de l'état caché
        hidden_states.append(h)

        # Sauvegarde de la sortie
        outputs.append(y)

    return hidden_states, outputs

# Fonction : cross_entropy()
#
# Cette fonction calcule l'erreur entre les mots prédits
# par le RNN et les mots attendus.
# Une erreur est calculée pour chaque mot de la phrase.
# Puis toutes les erreurs sont additionnées.
#
# Cette perte sera ensuite utilisée pour mettre à jour
# les poids du réseau.

def cross_entropy(outputs, target_sentence):

    # Découpage de la phrase anglaise
    target_words = target_sentence.split()

    # Initialisation de la perte
    loss = 0

    # Parcours des mots attendus
    for t, word in enumerate(target_words):

        # Vérifier que le mot existe dans le vocabulaire
        if word not in en_word_to_index:
            continue

        # Numéro du mot attendu
        index = en_word_to_index[word]

        # Probabilité prédite pour ce mot
        prediction = outputs[t]

        # Calcul de la Cross Entropy
        loss += -np.log(prediction[index, 0] + 1e-10)

    return loss

# Fonction : backward()
# Cette fonction réalise une version simplifiée de la Backpropagation Through Time (BPTT).
# Les poids de la couche de sortie ainsi que ceux de
# l'état caché sont progressivement corrigés.
learning_rate = 0.05

def backward(sentence, hidden_states, outputs, target_sentence):

    global Wxh, Whh, Whz, bh, bz

    input_words = sentence.split()
    target_words = target_sentence.split()

    # Gradient de l'état caché suivant
    dh_next = np.zeros((hidden_size, 1))

    # On parcourt la phrase à l'envers
    for t in reversed(range(len(target_words))):

        # Mot attendu
        target = np.zeros((output_size, 1))
        target[en_word_to_index[target_words[t]]] = 1

        # Erreur sur la sortie
        dy = outputs[t] - target

        # Mise à jour de Whz 
        Whz -= learning_rate * np.dot(dy, hidden_states[t].T)

        bz -= learning_rate * dy

        # Erreur sur l'état caché 
        dh = np.dot(Whz.T, dy) + dh_next

        # Dérivée de tanh
        dh_raw = (1 - hidden_states[t] ** 2) * dh

        # Mot d'entrée
        x = one_hot(fr_word_to_index[input_words[t]], input_size)

        # Etat caché précédent
        if t == 0:
            h_prev = np.zeros((hidden_size, 1))
        else:
            h_prev = hidden_states[t - 1]

        # Mise à jour de Wxh 
        Wxh -= learning_rate * np.dot(dh_raw, x.T)

        # Mise à jour de Whh -
        Whh -= learning_rate * np.dot(dh_raw, h_prev.T)

        # Mise à jour du biais caché 
        bh -= learning_rate * dh_raw

        # Gradient transmis au temps précédent
        dh_next = np.dot(Whh.T, dh_raw)


# Fonction : train()
#
# Cette fonction entraîne le réseau.
# Pour chaque phrase du jeu de données :
# 1. Le RNN réalise la propagation avant.
# 2. La fonction de perte est calculée.
# 3. Les poids sont mis à jour grâce à la
#    rétropropagation simplifiée.
# Ce processus est répété plusieurs fois (Epochs) afin d'améliorer progressivement les prédictions.
def train(epochs=500):
    print("Début de l'entraînement du RNN")

    for epoch in range(epochs):

        total_loss = 0

        # Parcours de toutes les phrases
        for french, english in dataset:

            # Propagation avant
            hidden_states, outputs = forward(french)

            # Calcul de la perte
            loss = cross_entropy(outputs, english)

            total_loss += loss

            # Mise à jour des poids
            backward(french, hidden_states, outputs, english)

        # Affichage toutes les 100 epochs
        if (epoch + 1) % 100 == 0:

            print(f"Epoch {epoch+1}/{epochs} | Loss = {total_loss:.4f}")

    print("\nEntraînement terminé.")

# Fonction : predict()
# Cette fonction teste le réseau sur une phrase.
#
# Pour chaque mot de la phrase française, le réseau prédit le mot anglais ayant la plus grande probabilité.
def predict(sentence):

    print("Test du réseau")
    # Propagation avant
    hidden_states, outputs = forward(sentence)

    predicted_words = []

    # Recherche du mot ayant la plus grande probabilité
    for output in outputs:

        index = np.argmax(output)

        predicted_words.append(index_to_en_word[index])

    print("Français :", sentence)
    print("Traduction prédite :", " ".join(predicted_words))

# Programme principal

train()

predict("bonjour")

predict("merci")

predict("chat noir")

Nombre de mots français : 14
Nombre de mots anglais  : 14
Début de l'entraînement du RNN
Epoch 100/500 | Loss = 6.2334
Epoch 200/500 | Loss = 0.4865
Epoch 300/500 | Loss = 0.2227
Epoch 400/500 | Loss = 0.1412
Epoch 500/500 | Loss = 0.1024

Entraînement terminé.
Test du réseau
Français : bonjour
Traduction prédite : hello
Test du réseau
Français : merci
Traduction prédite : thanks
Test du réseau
Français : chat noir
Traduction prédite : black cat
